# Data Preparation and Cleaning

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
# Or set to a specific large number (from my df)
pd.set_option('display.max_columns', 1163)

## Data Source (Education)
- https://unehistoireduconflitpolitique.fr/telecharger.html

#### Skill Share
(According to the mail from Prof. Loumeau)

**Skill Share** measures the proportion of highly educated individuals (holding a *baccalauréat* or higher) within a municipality.

$$\text{Skill Share} = \frac{\text{Educated Residents}}{\text{Total Residents}}$$

* **Example:** If a town has 1,000 residents and 200 hold a degree $\rightarrow \text{Skill Share} = 0.20 \text{ (20\%)}$.

---

#### Delta Skill Share
**Delta Skill Share** captures the temporal change in this educated fraction immediately surrounding an election cycle.

$$\Delta\text{Skill Share} = \text{Skill Share}_{\text{After Election}} - \text{Skill Share}_{\text{Before Election}}$$

* **Positive Value ($+0.03$):** The proportion of highly educated residents grew post-election.
* **Negative Value ($-0.03$):** The proportion of highly educated residents shrank post-election.

In [3]:
# Load education dat
df_edu = pd.read_csv("diplomescommunes.csv", dtype={"codecommune": str})

# Standardise commune code to 5 characters (leading zeros)
df_edu["codecommune"] = df_edu["codecommune"].str.zfill(5)

C:\Users\tobia\AppData\Local\Temp\ipykernel_8972\2857989072.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_edu = pd.read_csv("diplomescommunes.csv", dtype={"codecommune": str})


### Outcome Variable: Skill Share

| Column pattern | Meaning |
|---|---|
| `nodiph/f{year}` | Residents *without* any diploma (men/women) |
| `bach{year}/bacf{year}/bac{year}` | Residents with *baccalauréat* — high school level (men/women) |
| `suph{year}/supf{year}/sup{year}` | Residents with *university-level* education (men/women) |

**h** = hommes (men)

**f** = femmes (women)

**No info** = h + f = men + women

**Skill share** = (bach + sup) / (nodip + bach + sup), for men and women combined.  
**Delta skill share** = skill share *after* election − skill share *before* election.  
A negative value means the educated share *shrunk* post-election.

In [4]:
# Helper functions
def compute_skill_share(df, year):
    """Fraction of residents with bac or higher, for a given census year."""
    educated = (df[f"bach{year}"] + df[f"suph{year}"] +    # men: bac + university
                df[f"bacf{year}"] + df[f"supf{year}"])     # women: bac + university
    total    = (educated +
                df[f"nodiph{year}"] + df[f"nodipf{year}"]) # add no-diploma residents
    return educated / total                                 # share between 0 and 1

def compute_skill_share_trend(df, pre2_y, pre_y):
    """Change in skill share between two pre-treatment census years (pre-trend proxy)."""
    return compute_skill_share(df, pre_y) - compute_skill_share(df, pre2_y)

def compute_total_pop(df, year):
    """Approximate total population from the three education tiers."""
    return (df[f"nodiph{year}"] + df[f"nodipf{year}"] +
            df[f"bach{year}"]   + df[f"bacf{year}"]   +
            df[f"suph{year}"]   + df[f"supf{year}"])

def compute_gender_gap(df, year):
    """Tertiary-education share among men minus among women (pre-treatment year)."""
    tertiary_m = df[f"bach{year}"] + df[f"suph{year}"]
    tertiary_f = df[f"bacf{year}"] + df[f"supf{year}"]
    total_m = tertiary_m + df[f"nodiph{year}"]
    total_f = tertiary_f + df[f"nodipf{year}"]
    return (tertiary_m / total_m) - (tertiary_f / total_f)

def compute_no_diploma_share(df, year):
    """Share of residents with no diploma (pre-treatment year)."""
    no_diploma = df[f"nodiph{year}"] + df[f"nodipf{year}"]
    total = no_diploma + df[f"bach{year}"] + df[f"bacf{year}"] + df[f"suph{year}"] + df[f"supf{year}"]
    return no_diploma / total

In [5]:
# Each tuple: (election_year, pre2_census_year, pre_census_year, post_census_year)
# pre2 is added for the skill_share_trend covariate (needs two pre-treatment points).
# Uses a uniform 5-year lag before pre_census_year, consistent with the existing
# pre/post gap structure used in the 2014 and 2020 cycles 
# (see caveat above on interpolated annual values — a 5-year lag is chosen to approximate real
# census-to-census variation rather than interpolation noise).
windows = [
    (2008, 1994, 1999, 2009),
    (2014, 2004, 2009, 2014),
    (2020, 2009, 2014, 2019),
]

In [6]:
rows = []
for election_year, pre2_y, pre_y, post_y in windows:
    # Compute skill share at pre- and post-election census
    share_pre  = compute_skill_share(df_edu, pre_y)
    share_post = compute_skill_share(df_edu, post_y)
    # Delta = change in educated share around the election
    delta = share_post - share_pre

    # Carry population at both waves forward — filtering happens later, not here
    pop_pre  = compute_total_pop(df_edu, pre_y)
    pop_post = compute_total_pop(df_edu, post_y)

    # Build one df for this election year
    tmp = pd.DataFrame({
        "commune_id":          df_edu["codecommune"],
        "election_year":       election_year,
        "skill_share_pre":     share_pre,
        "skill_share_trend":   compute_skill_share_trend(df_edu, pre2_y, pre_y),
        "delta_skill_share":   delta,
        "pop_pre":             pop_pre,
        "pop_post":            pop_post,
        "gender_gap_tertiary": compute_gender_gap(df_edu, pre_y),
        "no_diploma_share":    compute_no_diploma_share(df_edu, pre_y),
    })
    rows.append(tmp)

df_outcome = pd.concat(rows, ignore_index=True)

In [7]:
df_outcome.head(3)

,commune_id,election_year,skill_share_pre,skill_share_trend,delta_skill_share,pop_pre,pop_post,gender_gap_tertiary,no_diploma_share
0,01001,2008,0.135593,0.021516,0.244295,472.0,537.0,-0.004598,0.864407
1,01002,2008,0.258065,0.018748,0.268882,124.0,167.0,0.112500,0.741935
2,01003,2008,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Source (French Electoral outcomes)
- https://www.data.gouv.fr/datasets/donnees-des-elections-agregees

In [8]:
# Load the full candidats file
# Large file low_memory=False prevents dtype warnings
df_cand = pd.read_csv("candidats_results.csv", sep=";", encoding="latin-1", low_memory=False)

In [9]:
# Standardise commune code to 5 characters (leading zeros)
df_cand["code_commune"] = df_cand["code_commune"].str.zfill(5)

In [10]:
# Inspect all unique nuance codes
print(df_cand["nuance"].unique())

['LDVC' 'LDVD' nan 'LDIV' 'LRN' 'LDVG' 'LUG' 'LUD' 'LCOM' 'LUXD' 'LLR'
 'LFI' 'LEXD' 'LEXG' 'LREG' 'LUC' 'LUDR' 'LSOC' 'LUDI' 'LECO' 'LVEC'
 'LHOR' 'LREN' 'LMDM' 'LREC' 'LDSV' 'DXG' 'DVG' 'DIV' 'REG' 'ECO' 'NUP'
 'RDG' 'ENS' 'LR' 'DVC' 'DSV' 'DVD' 'UDI' 'REC' 'RN' 'DXD' 'REM' 'MDM'
 'FN' 'SOC' 'COM' 'FI' 'EXD' 'DLF' 'EXG' 'NC' 'LNC' 'LREM' 'LDLF' 'LGJ'
 'BC-UCD' 'BC-UG' 'BC-RN' 'BC-UD' 'BC-UGE' 'BC-UXD' 'BC-LR' 'BC-DVD'
 'BC-DVG' 'BC-DVC' 'BC-DIV' 'BC-RDG' 'BC-SOC' 'BC-COM' 'BC-UC' 'BC-REM'
 'BC-UCG' 'BC-ECO' 'BC-UDI' 'BC-MDM' 'BC-EXD' 'BC-FI' 'BC-EXG' 'BC-REG'
 'BC-DSV' 'BC-GJ' 'LAUT' 'LMAJ' 'LGC' 'LMC' 'LCMD' 'LFN' 'UG' 'UXD' 'HOR'
 'VEC' 'LENS' 'LRDG' 'LVEG' 'LFG' 'LUGE' 'LUCD' 'UMP' 'MNR' 'PREP' 'MPF'
 'LO' 'LCR' 'CPNT' 'RPF' 'PRG' 'UDF' 'DL' 'LCOP' 'RPR' 'MDC' 'UDFD' 'M-NC'
 'AUT' 'MNA' 'GAU' 'FRN' 'DTE' 'LDV' 'LDD' 'LPC' 'LXD' 'LUDF' 'LUMP' 'LPS'
 'LCP' 'LRG' 'LXG' 'LVE' 'LDG' 'LEC' 'LGA' 'LDR' 'PG' 'M' 'MODM' 'BOVE'
 'BUFF' 'SCHI' 'LAGU' 'VILL' 'VOYN' 'BAYR' 'BESA' 'ROYA' 'SARK'

#### Defining far right code names 
| Code    | Full Name                       | Years Active          |
|---------|---------------------------------|-----------------------|
| RN      | Rassemblement National          | 2020, 2026            |
| LRN     | Liste Rassemblement National    | 2020, 2026            |
| BC-RN   | Binôme/coalition RN             | 2020, 2026            |
| FN      | Front National                  | 2008, 2014            |
| FRN     | Front National                  | 2008, 2014, 2020      |
| LFN     | Liste Front National            | 2008, 2014            |
| BC-FN   | Binôme FN                       | 2014                  |
| EXD     | Extrême droite (generic)        | all years             |
| LEXD    | Liste extrême droite            | all years             |
| BC-EXD  | Binôme extrême droite           | 2014, 2020            |
| DXD     | Divers extrême droite           | all years             |
| LUXD    | Liste union extrême droite      | 2020, 2026            |
| UXD     | Union extrême droite            | all years             |
| BC-UXD  | Binôme union extrême droite     | 2014, 2020            |
| MNR     | Mouvement National Républicain  | 2008                  |

In [11]:
# Keep only the columns needed, drop the rest as the df is too big otherwise
df_slim = df_cand[[
    "id_election",    # year + round identifier
    "code_commune",   # municipality code
    "nuance",         # party affiliation
    "voix",           # vote count
]].copy()

In [12]:
df_slim.head(3)

,id_election,code_commune,nuance,voix
0,2026_muni_t2,01160,LDVC,257
1,2026_muni_t2,01160,LDVC,311
2,2026_muni_t2,01160,LDVC,252


In [13]:
# Parse year and round and add them as new columns
df_slim["election_year"] = df_slim["id_election"].str.split("_").str[0].astype(int)
df_slim["tour"]          = df_slim["id_election"].str.split("_").str[2]

# Keep only municipal elections
df_slim = df_slim[df_slim["id_election"].str.contains("muni")].drop(columns="id_election")

# Export, hence create a new file with only the columns needed for the RDD analysis
df_slim.to_csv("candidats_slim.csv", index=False)

In [14]:
df_slim.head(3)

,code_commune,nuance,voix,election_year,tour
0,01160,LDVC,257,2026,t2
1,01160,LDVC,311,2026,t2
2,01160,LDVC,252,2026,t2


In [15]:
# Define far-right nuance codes
far_right_nuances = [
    "RN", "LRN", "BC-RN",           # Rassemblement National (2020, 2026)
    "FN", "LFN", "BC-FN", "FRN"     # Front National (2008, 2014)
    "EXD", "LEXD", "BC-EXD",        # Extrême droite generic
    "DXD", "LUXD", "UXD", "BC-UXD", # Union / divers extrême droite
    "MNR",                          # Mégret splinter (2008)
]

# Possible that not all these nuances are present in the dataset, but we can still flag them if they are.

# Flag far-right rows
df_slim["is_far_right"] = df_slim["nuance"].isin(far_right_nuances)

In [16]:
# I do not use data from year 2026 right now
df_slim = df_slim[df_slim["election_year"] != 2026]

In [17]:
# Calculate share of far-right rows
far_right_share = df_slim["is_far_right"].mean() * 100
print(f"Share of far-right candidates: {far_right_share:.2f}%")

# Also show counts
print(f"\nCounts:")
print(df_slim["is_far_right"].value_counts())
print(f"\nTotal rows: {len(df_slim):,}")

Share of far-right candidates: 2.69%

Counts:
is_far_right
False    1424106
True       39424
Name: count, dtype: int64

Total rows: 1,463,530


In [18]:
# Determine the decisive round per commune/year
#    (max tour number: 2 > 1)
df_slim['tour_num'] = df_slim['tour'].str.extract('(\d+)').astype(int)

# Get the highest tour number for each (commune, year)
decisive_rounds = (
    df_slim.groupby(['code_commune', 'election_year'])['tour_num']
    .max()
    .reset_index(name='decisive_tour')
)

# Keep only rows that belong to that decisive round
df_dec = df_slim.merge(decisive_rounds, on=['code_commune', 'election_year'], how='inner')
df_dec = df_dec[df_dec['tour_num'] == df_dec['decisive_tour']]

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\tobia\AppData\Local\Temp\ipykernel_8972\2142661507.py:3: SyntaxWarning: invalid escape sequence '\d'
  df_slim['tour_num'] = df_slim['tour'].str.extract('(\d+)').astype(int)


In [19]:
# Identify the winner in each decisive round
#    (row with maximum voix)
winner_idx = df_dec.groupby(['code_commune', 'election_year'])['voix'].idxmax()
df_winners = df_dec.loc[winner_idx].copy()

In [20]:
df_winners.head(3)

,code_commune,nuance,voix,election_year,tour,is_far_right,tour_num,decisive_tour
771511,01001,NC,331,2014,t1,False,1,1
452703,01001,NC,267,2020,t1,False,1,1
771524,01002,NC,165,2014,t1,False,1,1


In [21]:
# ── Recreate treat column (1 = far-right won the decisive round, 0 = otherwise) ──
# df_winners should already contain only ONE row per commune x election_year
# (the winning list, selected via idxmax on voix). If so, is_far_right IS treat.
df_winners['treat'] = df_winners['is_far_right'].astype(int)

# Sanity check: one row per commune x election_year, and reasonable win count
print(df_winners.groupby(['code_commune', 'election_year']).size().max())  # should be 1
print(df_winners['treat'].sum())  # should match your earlier count (26)

1
26


When does a winner get <50%?

In French municipal elections, when 3+ lists contest the decisive round, votes split across them. The winner only needs a plurality (most votes), not an absolute majority (>50%). Example: three lists get 40%/35%/25% — the 40% list wins despite most voters (60%) voting against it.

This happens routinely in French municipal races (multi-list contests are common, especially in larger communes), so a meaningful share of your winners will be below 50%.

In [22]:
# creating the d_50 variable (plurality margin) for each commune x year, in percentage points

# Step 1: total votes cast per commune x year (decisive round only) 
total_votes = (
    df_dec.groupby(['code_commune', 'election_year'])['voix']
    .sum()
    .reset_index(name='voix_total')
)

# Step 2: far-right votes per commune x year (0 if no far-right list ran) 
# (if multiple far-right lists ran in one commune, they get added together).
far_right_votes = (
    df_dec[df_dec['is_far_right']]
    .groupby(['code_commune', 'election_year'])['voix']
    .max() # get the maximum votes for any far-right list in the decisive round
    .reset_index(name='voix_far_right')
)

# Step 3: best NON-far-right list's votes per commune x year
# (this is who the far-right actually had to beat)
runner_up_votes = (
    df_dec[~df_dec['is_far_right']]
    .groupby(['code_commune', 'election_year'])['voix']
    .max() # instead of sum, we want the best rival's votes
    .reset_index(name='voix_runner_up')
)

# Step 4: merge total votes, far-right votes, and runner-up votes into one table
margin_table = (
    total_votes # is the base table, since it contains all commune x year combinations
    .merge(far_right_votes, on=['code_commune', 'election_year'], how='left')
    .merge(runner_up_votes, on=['code_commune', 'election_year'], how='left')
    .fillna(0)  # communes with no far-right list -> voix_far_right = 0
)

# Step 5: compute d_50 as the plurality margin, in percentage points
# (far-right share − best-rival share) x 100
margin_table['d_50'] = (
    (margin_table['voix_far_right'] - margin_table['voix_runner_up']) / margin_table['voix_total']
) * 100 # as percentage points

# Step 6: drop any stale d_50 in df_winners, then merge in the new one cleanly
df_winners = df_winners.drop(columns=['d_50'], errors='ignore').merge(
    margin_table[['code_commune', 'election_year', 'd_50']],
    on=['code_commune', 'election_year'],
    how='left'
)

In [23]:
df_winners.head(3)

,code_commune,nuance,voix,election_year,tour,is_far_right,tour_num,decisive_tour,treat,d_50
0,01001,NC,331,2014,t1,False,1,1,0,-7.090831
1,01001,NC,267,2020,t1,False,1,1,0,-6.964006
2,01002,NC,165,2014,t1,False,1,1,0,-9.939759


## Merge electoral with educational Data

In [24]:
df_winners.head(3)

,code_commune,nuance,voix,election_year,tour,is_far_right,tour_num,decisive_tour,treat,d_50
0,01001,NC,331,2014,t1,False,1,1,0,-7.090831
1,01001,NC,267,2020,t1,False,1,1,0,-6.964006
2,01002,NC,165,2014,t1,False,1,1,0,-9.939759


In [25]:
df_outcome.tail(3)

,commune_id,election_year,skill_share_pre,skill_share_trend,delta_skill_share,pop_pre,pop_post,gender_gap_tertiary,no_diploma_share
113952,95680,2020,0.345784,0.046966,0.056384,16545.0,16513.0,-0.020387,0.654216
113953,95682,2020,0.320000,-0.013333,-0.037949,125.0,156.0,-0.314103,0.680000
113954,95690,2020,0.648000,0.114102,0.144453,250.0,265.0,-0.064000,0.352000


In [26]:
# 1. Standardise both keys to 5-digit strings (this fixes the type mismatch)
df_winners['code_commune'] = df_winners['code_commune'].astype(str).str.zfill(5)
df_outcome['commune_id'] = df_outcome['commune_id'].astype(str).str.zfill(5)

In [27]:
# 2. Merge on both commune and year
df_merged = df_winners.merge(
    df_outcome,
    how='left',  # Keep all winners, even if outcome is missing
    left_on=['code_commune', 'election_year'],
    right_on=['commune_id', 'election_year']
)

In [28]:
# 3. sanity check
print(f"Total winners: {len(df_winners)}")
print(f"Rows after merge: {len(df_merged)}")
print(f"Rows with missing outcome: {df_merged['delta_skill_share'].isna().sum()}")

Total winners: 74370
Rows after merge: 74370
Rows with missing outcome: 1857


In [29]:
df_merged.head(3)

,code_commune,nuance,voix,election_year,tour,is_far_right,tour_num,decisive_tour,treat,d_50,commune_id,skill_share_pre,skill_share_trend,delta_skill_share,pop_pre,pop_post,gender_gap_tertiary,no_diploma_share
0,01001,NC,331,2014,t1,False,1,1,0,-7.090831,01001,0.379888,0.094717,0.040776,537.0,542.0,-0.002456,0.620112
1,01001,NC,267,2020,t1,False,1,1,0,-6.964006,01001,0.420664,0.040776,0.017279,542.0,564.0,-0.015528,0.579336
2,01002,NC,165,2014,t1,False,1,1,0,-9.939759,01002,0.526946,0.145002,0.084482,167.0,175.0,-0.024412,0.473054


In [30]:
# Drop unnecessary columns (tour_num, decisive_tour, commune_id) after the merge
df_merged = df_merged.drop(columns=["tour_num", "decisive_tour", "commune_id"])

In [31]:
# Quick check of treat dummy distribution
df_merged['treat'].value_counts()

treat
0    74344
1       26
Name: count, dtype: int64

In [32]:
# Drop al entries with no far-right participation in the decisive round
# I only need rounds where the far-right actually fielded a list, otherwise the RDD is not well-defined.

# Add a participation flag (True if far-right fielded any list in the decisive round)
participation = (
    df_dec.groupby(['code_commune', 'election_year'])['is_far_right']
    .any()
    .reset_index(name='far_right_participated')
)

# add to df_merged
df_merged = df_merged.merge(
    participation, on=['code_commune', 'election_year'], how='left'
)

# Filter to far-right participants only (drop the rest)
df_rdd = df_merged[df_merged['far_right_participated']].copy()
print(f"N observations in RDD sample: {len(df_rdd)}")
print(f"N far-right wins (treat=1): {df_rdd['treat'].sum()}")

N observations in RDD sample: 843
N far-right wins (treat=1): 26


In [33]:
# Filter out communes with very small populations (pre- or post-election)
min_pop = 100
df_rdd = df_rdd[(df_rdd['pop_pre'] >= min_pop) & (df_rdd['pop_post'] >= min_pop)].copy()

In [34]:
df_rdd

,code_commune,nuance,voix,election_year,tour,is_far_right,treat,d_50,skill_share_pre,skill_share_trend,delta_skill_share,pop_pre,pop_post,gender_gap_tertiary,no_diploma_share,far_right_participated
102,01053,LUG,437,2014,t1,False,0,-2.409639,0.384403,0.028441,0.022447,27505.0,28030.0,-0.000504,0.615597,True
103,01053,LUG,388,2020,t1,False,0,-4.088756,0.406850,0.022447,0.022403,28030.0,28927.0,-0.005866,0.593150,True
1168,02168,LDVG,335,2014,t1,False,0,-5.130327,0.322229,0.049262,0.016947,9906.0,10098.0,0.005124,0.677771,True
1169,02168,LDVC,264,2020,t1,False,0,-5.420827,0.339176,0.016947,-0.009506,10098.0,10374.0,-0.031530,0.660824,True
1179,02173,LDVD,315,2014,t2,False,0,-4.599702,0.275587,0.049451,0.039119,8266.0,8160.0,0.022809,0.724413,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73710,95491,LDVD,386,2014,t1,False,0,-8.980798,0.500472,0.061387,0.044734,5293.0,5497.0,-0.017223,0.499528,True
73747,95563,LUMP,411,2014,t1,False,0,-5.092448,0.537022,0.048996,0.034063,10048.0,10403.0,0.002764,0.462978,True
73777,95607,LUG,344,2008,t1,False,0,-3.158237,0.368726,0.024197,0.111584,16576.0,17293.0,0.007391,0.631274,True
73778,95607,LUMP,462,2014,t2,False,0,-3.745754,0.480310,0.049311,0.035686,17293.0,17504.0,0.009470,0.519690,True


## Add Economic Data

**Data Sources:**

**Insee** Institut national de la statistique et des études économiques, declared income (DEC)

- Data from 2008: https://www.insee.fr/fr/statistiques/1893295?sommaire=2389000
- Data from 2014: https://www.insee.fr/fr/statistiques/3288151
- Data from 2020: https://www.insee.fr/fr/statistiques/7233950 

In [35]:
# Commune list you actually need (from df_rdd)
communes_needed = set(df_rdd['code_commune'].astype(str).str.zfill(5))

In [36]:
# import 2008 dataframe
# RFDU2008COM.xls (commune-level, 2008)
df_2008 = pd.read_excel(
    'RFDU2008COM.xls',
    sheet_name='D_UC',
    header=6  # real header is on row 7 (0-indexed row 6)
)

df_2008['COM'] = df_2008['COM'].astype(str).str.zfill(5)
df_2008 = df_2008[df_2008['COM'].isin(communes_needed)]
df_2008 = df_2008[['COM', 'RFUCQ208', 'RFUCGI08']]  # keep median + gini only
df_2008 = df_2008.rename(columns={'COM': 'code_commune', 'RFUCQ208': 'income_median_2008', 'RFUCGI08': 'income_gini_2008'})

In [37]:
# Create function to load IRIS income data for 2014 and 2020, aggregate to commune level, and handle French decimal commas + suppression markers ("ns", "nd").
# IRIS files (2014 & 2020) — need IRIS → commune aggregation
def load_iris_income(path, sheet, header_row, med_col, gini_col, year):
    df = pd.read_excel(path, sheet_name=sheet, header=header_row)
    df['COM'] = df['COM'].astype(str).str.zfill(5)
    df = df[df['COM'].isin(communes_needed)]

    # Fix French comma decimals and confidentiality markers ("ns", "nd") if present
    for col in [med_col, gini_col]:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
            df[col] = pd.to_numeric(df[col], errors='coerce')  # "ns"/"nd" -> NaN instead of crashing

    # Aggregate IRIS -> commune (simple mean for now, see caveat below)
    df_com = (
        df.groupby('COM')[[med_col, gini_col]].mean()
        .reset_index()
        .rename(columns={
            'COM': 'code_commune',
            med_col: f'income_median_{year}',
            gini_col: f'income_gini_{year}',
        })
    )
    return df_com

In [38]:
#Import data, run the function for 2014 and 2020

# 2014 — declared income (DEC), comparable to RFL 2008's "revenu fiscal" concept
df_2014 = load_iris_income(
    'BASE_TD_FILO_DEC_IRIS_2014.xls', 'IRIS_DEC',
    header_row=5, med_col='DEC_MED14', gini_col='DEC_GI14', year=2014
)

# 2020 — declared income (DEC)
df_2020 = load_iris_income(
    'BASE_TD_FILO_DEC_IRIS_2020.xlsx', 'IRIS_DEC',
    header_row=5, med_col='DEC_MED20', gini_col='DEC_GI20', year=2020
)

In [39]:
# Merge and standardize the income dataframes for 2008, 2014, and 2020 into df_rdd

# Add election_year to each df and standardize column names ──
df_2008 = df_2008.rename(columns={'income_median_2008': 'income_median', 'income_gini_2008': 'income_gini'})
df_2008['election_year'] = 2008

df_2014 = df_2014.rename(columns={'income_median_2014': 'income_median', 'income_gini_2014': 'income_gini'})
df_2014['election_year'] = 2014

df_2020 = df_2020.rename(columns={'income_median_2020': 'income_median', 'income_gini_2020': 'income_gini'})
df_2020['election_year'] = 2020

# Stack into one long df ──
df_income = pd.concat([df_2008, df_2014, df_2020], ignore_index=True)

# Merge into df_rdd on code_commune + election_year ──
df_rdd = df_rdd.merge(df_income, on=['code_commune', 'election_year'], how='left')

In [40]:
df_rdd

,code_commune,nuance,voix,election_year,tour,is_far_right,treat,d_50,skill_share_pre,skill_share_trend,delta_skill_share,pop_pre,pop_post,gender_gap_tertiary,no_diploma_share,far_right_participated,income_median,income_gini
0,01053,LUG,437,2014,t1,False,0,-2.409639,0.384403,0.028441,0.022447,27505.0,28030.0,-0.000504,0.615597,True,17694.500000,0.365057
1,01053,LUG,388,2020,t1,False,0,-4.088756,0.406850,0.022447,0.022403,28030.0,28927.0,-0.005866,0.593150,True,18276.111111,0.375722
2,02168,LDVG,335,2014,t1,False,0,-5.130327,0.322229,0.049262,0.016947,9906.0,10098.0,0.005124,0.677771,True,17063.666667,0.368367
3,02168,LDVC,264,2020,t1,False,0,-5.420827,0.339176,0.016947,-0.009506,10098.0,10374.0,-0.031530,0.660824,True,17170.000000,0.387833
4,02173,LDVD,315,2014,t2,False,0,-4.599702,0.275587,0.049451,0.039119,8266.0,8160.0,0.022809,0.724413,True,15662.666667,0.399675
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
832,95491,LDVD,386,2014,t1,False,0,-8.980798,0.500472,0.061387,0.044734,5293.0,5497.0,-0.017223,0.499528,True,NaN,NaN
833,95563,LUMP,411,2014,t1,False,0,-5.092448,0.537022,0.048996,0.034063,10048.0,10403.0,0.002764,0.462978,True,28190.666667,0.309148
834,95607,LUG,344,2008,t1,False,0,-3.158237,0.368726,0.024197,0.111584,16576.0,17293.0,0.007391,0.631274,True,22209.000000,0.309100
835,95607,LUMP,462,2014,t2,False,0,-3.745754,0.480310,0.049311,0.035686,17293.0,17504.0,0.009470,0.519690,True,23755.090909,0.303520


In [41]:
# Break missingness down by year AND by variable separately
print(df_rdd.groupby('election_year')[['income_median', 'income_gini']].apply(lambda x: x.isna().sum()))

# And total rows per year, for context
print(df_rdd['election_year'].value_counts())

               income_median  income_gini
election_year                            
2008                       0            0
2014                     168          168
2020                      29           29
election_year
2014    513
2020    285
2008     39
Name: count, dtype: int64


"FiLoSoFi's declared-income (DEC) product had substantially narrower national commune coverage in its 2014 vintage (969 communes) compared to 2020 (1,887 communes), resulting in 225 sample communes (168 commune-election observations) lacking 2014 income data. This asymmetry is disclosed in the balance-check results by reporting the effective N for each election cycle rather than assuming complete coverage."

## Add Unemployment Data

**Data Sources:**

**Insee** Institut national de la statistique et des études économiques, Employment - working population

- Data from 2008: https://www.insee.fr/fr/statistiques/2044654
- Data from 2014: https://www.insee.fr/fr/statistiques/2862207
- Data from 2020: https://www.insee.fr/fr/statistiques/7632867

Unemployment comuptation: **CHOM / ACT**
- total **active** pop. (15-64yrs): P08_**ACT**1564 (2008)
- total **unemployed** pop. (15-64yrs): P08_**CHOM**1564 (2008)

In [42]:
# function to load employment data, compute unemployment rate, and standardize commune codes.

# Same defensive handling as income (French decimal commas + "ns"/"nd" suppression markers) in case they appear.
def load_unemployment(path, sheet, header_row, act_col, chom_col, year): # navigate to the correct sheet, header row, and column names for each year
    df = pd.read_excel(path, sheet_name=sheet, header=header_row)
    df['CODGEO'] = df['CODGEO'].astype(str).str.zfill(5)
    df = df[df['CODGEO'].isin(communes_needed)] # only keep communes that are actually in the RDD sample

    # Fix French comma decimals and confidentiality markers ("ns", "nd") if present
    for col in [act_col, chom_col]:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.replace(',', '.', regex=False)
            df[col] = pd.to_numeric(df[col], errors='coerce')  # "ns"/"nd" -> NaN instead of crashing

    # Unemployment rate = chômeurs / actifs (15-64), in %
    df['unemployment_rate'] = (df[chom_col] / df[act_col]) * 100

    df_com = df[['CODGEO', 'unemployment_rate']].rename(columns={'CODGEO': 'code_commune'})
    df_com['election_year'] = year
    return df_com

In [43]:
# 2008
df_unemp_2008 = load_unemployment(
    'base-cc-emploi-pop-active-2008.xls', 'COM_2008',
    header_row=5, act_col='P08_ACT1564', chom_col='P08_CHOM1564', year=2008
)

# 2014
df_unemp_2014 = load_unemployment(
    'base-cc-emploi-pop-act-2014.xls', 'COM_2014',
    header_row=5, act_col='P14_ACT1564', chom_col='P14_CHOM1564', year=2014
)

# 2020
df_unemp_2020 = load_unemployment(
    'base-cc-emploi-pop-active-2020_v2.xlsx', 'COM_2020',
    header_row=5, act_col='P20_ACT1564', chom_col='P20_CHOM1564', year=2020
)

In [44]:
# Stack into one long df, matching the income_df pattern
df_unemployment = pd.concat([df_unemp_2008, df_unemp_2014, df_unemp_2020], ignore_index=True)

# Merge into df_rdd on code_commune + election_year
df_rdd = df_rdd.merge(df_unemployment, on=['code_commune', 'election_year'], how='left')

In [45]:
df_rdd

,code_commune,nuance,voix,election_year,tour,is_far_right,treat,d_50,skill_share_pre,skill_share_trend,delta_skill_share,pop_pre,pop_post,gender_gap_tertiary,no_diploma_share,far_right_participated,income_median,income_gini,unemployment_rate
0,01053,LUG,437,2014,t1,False,0,-2.409639,0.384403,0.028441,0.022447,27505.0,28030.0,-0.000504,0.615597,True,17694.500000,0.365057,17.997301
1,01053,LUG,388,2020,t1,False,0,-4.088756,0.406850,0.022447,0.022403,28030.0,28927.0,-0.005866,0.593150,True,18276.111111,0.375722,17.078005
2,02168,LDVG,335,2014,t1,False,0,-5.130327,0.322229,0.049262,0.016947,9906.0,10098.0,0.005124,0.677771,True,17063.666667,0.368367,22.861113
3,02168,LDVC,264,2020,t1,False,0,-5.420827,0.339176,0.016947,-0.009506,10098.0,10374.0,-0.031530,0.660824,True,17170.000000,0.387833,23.566623
4,02173,LDVD,315,2014,t2,False,0,-4.599702,0.275587,0.049451,0.039119,8266.0,8160.0,0.022809,0.724413,True,15662.666667,0.399675,24.051101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
832,95491,LDVD,386,2014,t1,False,0,-8.980798,0.500472,0.061387,0.044734,5293.0,5497.0,-0.017223,0.499528,True,NaN,NaN,8.661228
833,95563,LUMP,411,2014,t1,False,0,-5.092448,0.537022,0.048996,0.034063,10048.0,10403.0,0.002764,0.462978,True,28190.666667,0.309148,10.062158
834,95607,LUG,344,2008,t1,False,0,-3.158237,0.368726,0.024197,0.111584,16576.0,17293.0,0.007391,0.631274,True,22209.000000,0.309100,8.712611
835,95607,LUMP,462,2014,t2,False,0,-3.745754,0.480310,0.049311,0.035686,17293.0,17504.0,0.009470,0.519690,True,23755.090909,0.303520,10.358991


## Export Final Dataframe

In [46]:
# Export clean df_rdd to CSV
df_rdd.to_csv('df_rdd_clean.csv', index=False)